# Bài 15 · RNN & LSTM: mạng có trí nhớ, từ nhớ một ký hiệu đến làm thơ lục bát

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/15-rnn.ipynb)

Notebook đi kèm [Bài 15 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/rnn/) (demo tương tác nằm ở trang bài học).
Một mạng hồi tiếp đọc chuỗi từng phần tử và mang theo một trạng thái ẩn $h$. Nó nhớ được bao xa? Ta tự viết RNN, GRU
và LSTM bằng numpy, đo gradient đi ngược qua thời gian trên đúng các chuỗi của demo, rồi chạy ba mô hình ngôn ngữ đã
học Truyện Kiều để xem chúng có nhớ được nhịp 6–8 của thơ lục bát không.

**Bạn sẽ làm:**
1. Viết một ô RNN và chạy nó từng bước, với cùng một bộ trọng số ở mọi bước.
2. Tự viết lan truyền ngược theo thời gian (BPTT) và kiểm tra nó bằng sai phân hữu hạn.
3. Đo phần gradient về tới bước đầu của chuỗi: RNN co hoặc bùng nổ, LSTM và GRU giữ được nhờ cổng, nhưng chỉ khi
   khởi tạo đúng.
4. Huấn luyện thật bài toán “nhớ ký hiệu đầu tiên”, rồi đọc kết quả quét đầy đủ của demo.
5. Nạp ba mô hình mức ký tự đã học Truyện Kiều, sinh thơ, và đo xem mạng nào nhớ được độ dài của câu trước.

**Cần biết trước:** [Bài 14 · Token & embedding](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/token-embedding/)
(một token thành một vectơ) và [Bài 09 · Gradient biến mất & khởi tạo](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/vanishing-gradient/)
(tích của nhiều thừa số, cắt ngưỡng gradient). Lượt ngược dạng ma trận là của
[Bài 06](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/lan-truyen-nguoc/), Adam của
[Bài 07](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/bo-toi-uu/), kết nối tắt của
[Bài 13](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/resnet/).
**Thời gian chạy:** dưới 1 phút trên CPU, kể cả tải dữ liệu lần đầu (Truyện Kiều ~150 KB và hai tệp kết quả ~570 KB).

Mọi con số trong notebook đều do chính các ô bên dưới tính ra, và trùng với số trong demo. Riêng các lần huấn luyện dài
(90 lần cho bài toán nhớ, 45 lần cho Truyện Kiều) được nạp từ đúng tệp mà demo dùng; notebook chạy lại một phần của
chúng và kiểm tra rằng ra cùng con số.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân,
dấu chấm hàng nghìn. Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi. Gradient ở bài này
có thể nhỏ tới $10^{-25}$, nên `p10(e)` in $10^e$ như demo: số thường trong khoảng $[10^{-3}, 10^4)$, ngoài khoảng đó dạng
$3{,}7 \cdot 10^{-8}$.

In [ ]:
import hashlib
import itertools
import json
import math
import random
import re
import unicodedata
import urllib.request
from base64 import b64decode
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm một mã hóa thứ hai (hình dạng, nét gạch, nhãn)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("cam_xam_xanh", [ORANGE, "#f2f2f2", BLUE])  # âm → 0 → dương
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38 (không in “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def p10(e):
    """10^e như demo: 0,23 · 38,6 · 3,7·10⁻⁸ (số thường từ 0,001 đến 10.000)."""
    v = 10.0**e
    if 1e-3 <= v < 1e4:
        return vn(v, 4 if v < 0.01 else 3 if v < 1 else 1 if v < 100 else 0)
    k = math.floor(e)
    m = 10 ** (e - k)
    if round(m * 10) >= 100:
        m, k = m / 10, k + 1
    return f"{vn(m, 1)}·10{str(k).translate(SUP)}"


def vn_tick(v, _pos=None):
    """Nhãn vạch trên trục: 0,5 · −2 · 60.000 (bỏ số 0 thừa ở cuối)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    """Nhãn vạch trục log: 0,5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return vn_tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{vn_tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def vn_axes(fig):
    """Đổi nhãn mọi trục của hình (cả thanh màu) sang kiểu Việt Nam; gọi ngay trước plt.show().
    Trục log lấy vạch 1–2–5 nếu ngắn, hoặc các lũy thừa của 10 (cách đều, tối đa ~9 vạch) nếu dài."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(vn_tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


print(vn(104204, 0), vn(0.8, 1), pct(0.7835), p10(-7.43), p10(2.342), p10(-0.07))

## 2 · Dữ liệu

Bài này có hai nguồn dữ liệu.

- **Truyện Kiều** của Nguyễn Du (phạm vi công cộng), lấy từ Wikisource tiếng Việt, đúng **bản sửa 205132** mà demo
  dùng. Ô dưới tải văn bản gốc (mã wiki) và kiểm tra MD5, nên ai chạy cũng có đúng cùng một văn bản. Wikimedia từ chối
  tên chương trình (User-Agent) mặc định của Python, nên ô gắn một tên riêng trước khi tải.
- **Hai tệp kết quả** mà demo dùng (`fetch_json`, có kiểm tra MD5): `rnn-memory.json`, kết quả quét bài toán “nhớ ký
  hiệu đầu tiên” (6 loại mạng × 5 độ dài × 3 seed), và `rnn-kieu.json`, ba mô hình ngôn ngữ đã học Truyện Kiều (trọng
  số int8) cùng kết quả đo của mọi lần huấn luyện. Chạy lại mỗi tệp mất hàng chục phút trên một máy nhiều nhân.

In [ ]:
KIEU_URL = "https://vi.wikisource.org/w/index.php?oldid=205132&action=raw"  # bản sửa 205132, mã wiki thô
KIEU_MD5 = "d4090daac094841888ee2b2b18661065"
opener = urllib.request.build_opener()
opener.addheaders = [("User-Agent", "HocSauReels-notebook/1.0 (https://d3qr1cjln6qqtk.cloudfront.net/)")]
urllib.request.install_opener(opener)  # với User-Agent mặc định của Python, Wikimedia trả lỗi 403


def fetch_kieu(path=Path("truyen_kieu_205132.wiki")):
    if not path.exists():
        urllib.request.urlretrieve(KIEU_URL, path)
    raw = path.read_bytes()
    assert hashlib.md5(raw).hexdigest() == KIEU_MD5, f"{path} bị hỏng, hãy xóa và tải lại"
    return raw


# Kết quả của một lần huấn luyện dài (cùng tệp với demo trên web). build.py thay COURSE_DATA_URL bằng thư mục data/
# của repo notebook công khai.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


raw = fetch_kieu()
MEM = fetch_json("rnn-memory.json", "dec97dd7970ad792aee3564a845c2d39")  # bài toán nhớ: kết quả quét của demo
KIEU = fetch_json("rnn-kieu.json", "8f2fbe5cb77248254eb9f07770651618")  # Truyện Kiều: mô hình và kết quả đo
print(f"mã wiki: {vn(len(raw), 0)} byte · hai tệp kết quả đã nạp")

Mã wiki có số thứ tự câu dạng `{{số|…}}` và vài dòng mô tả; bài thơ nằm giữa hai thẻ `<poem>`. Ta làm sạch đúng như
script của demo (`reel/rnn/kieu.py`) và kiểm tra MD5 của văn bản sạch. Văn bản được đọc ở **mức ký tự**: mỗi ký tự
(chữ có dấu, dấu câu, dấu cách, xuống dòng) là một token, như các mô hình của Karpathy (2015). 3.000 câu đầu để huấn
luyện, các câu còn lại là tập kiểm định mà mô hình chưa từng thấy.

In [ ]:
body = raw.decode("utf-8").split("<poem>")[1].split("</poem>")[0]
lines_raw = [re.sub(r"\{\{số\|\d+\}\}", "", ln).strip() for ln in body.split("\n")]  # bỏ số thứ tự câu
TEXT = "\n".join(ln for ln in lines_raw if ln) + "\n"
assert hashlib.md5(TEXT.encode()).hexdigest() == "dd94ee390bfc543287b11af62b2f2fc8", "khác với văn bản của demo"
LINES = TEXT[:-1].split("\n")
CHARS = sorted(set(TEXT))  # bảng ký tự: 132 token
V = len(CHARS)
IX = {ch: i for i, ch in enumerate(CHARS)}
IDS = np.array([IX[ch] for ch in TEXT])
N_TRAIN_LINES, WARM = 3000, 512
CUT = len("\n".join(LINES[:N_TRAIN_LINES])) + 1
TRAIN, VAL = IDS[:CUT], IDS[CUT:]
VAL_LINES = LINES[N_TRAIN_LINES:]
print(f"{vn(len(LINES), 0)} câu, {vn(len(TEXT), 0)} ký tự, {V} ký tự khác nhau")
print(
    f"huấn luyện: {vn(N_TRAIN_LINES, 0)} câu = {vn(len(TRAIN), 0)} ký tự · kiểm định: {len(VAL_LINES)} câu = {vn(len(VAL), 0)} ký tự"
)
print("bốn câu đầu:", *LINES[:4], sep="\n  ")
assert (len(LINES), len(TEXT), V) == (KIEU["source"]["lines"], KIEU["source"]["chars"], len(KIEU["chars"]))
assert "".join(CHARS) == KIEU["chars"] and VAL_LINES == KIEU["valLines"] and len(TRAIN) == KIEU["nTrain"]

## 3 · Một ô RNN: đọc từng ký hiệu, mang theo một trạng thái

Bắt đầu bằng bài toán nhỏ nhất cần trí nhớ, cũng là bài toán của demo: **“nhớ ký hiệu đầu tiên”**. Mỗi chuỗi gồm $T$
ký hiệu lấy ngẫu nhiên trong {A, B, C, D}, rồi dấu “?”. Mạng phải trả lời ký hiệu **đầu tiên**. Các ký hiệu ở giữa
chỉ là nhiễu, cùng bộ ký hiệu với đáp án, nên mạng phải giữ ký hiệu đầu suốt $T$ bước. Đoán mò đúng 25%.

Mạng hồi tiếp đơn giản (Elman, 1990) đọc ký hiệu $x_t$ ở bước $t$ và cập nhật trạng thái ẩn:

$$h_t = \tanh(x_t W_x + h_{t-1} W_h + b), \qquad h_{-1} = 0, \qquad \hat y = \operatorname{softmax}(h_T W_y + b_y)$$

- $x_t$ là vectơ one-hot của ký hiệu, nên $x_t W_x$ chỉ là một hàng của $W_x$: đúng là embedding của Bài 14.
- $W_x$, $W_h$, $b$ **giống nhau ở mọi bước**. Số tham số không phụ thuộc độ dài chuỗi.
- Chỉ trạng thái cuối $h_T$ (sau dấu “?”) được đọc ra, qua một lớp softmax 4 lớp (Bài 04).

Để trọng số trùng **từng bit** với demo, ta rút chúng đúng như trình duyệt: bộ sinh `mulberry32` của JavaScript, rồi
Box–Muller biến mỗi cặp số đều thành một số $\mathcal N(0, 1)$ (như Bài 09). Các số lấp lần lượt $W_x$, $W_h$, $W_y$;
$W_x \sim \mathcal N(0, 1/V)$, $W_h \sim \mathcal N(0, g^2/H)$ với hệ số $g = 1$ mặc định, $W_y \sim \mathcal N(0, 1/H)$,
các độ lệch bằng 0. Seed 0 của demo dùng `mulberry32(1)`.

In [ ]:
GATES = {"rnn": 1, "gru": 3, "lstm": 4}  # số khối cột của W: RNN 1; GRU r, z, h̃; LSTM i, f, o, c̃


def sig(z):
    """Hàm sigmoid σ(z), viết qua tanh để không tràn số khi z rất âm."""
    return 0.5 * (1.0 + np.tanh(0.5 * z))


def mulberry32(seed, n):
    """n số đều đầu tiên trong [0, 1) của mulberry32(seed), bộ sinh mà demo (JavaScript) dùng, viết dạng vectơ.
    Mọi phép tính là số nguyên 32 bit (phép nhân uint32 tự quay vòng như Math.imul), nên trùng từng bit với JS."""
    a = ((seed + np.arange(1, n + 1, dtype=np.uint64) * 0x6D2B79F5) & 0xFFFFFFFF).astype(np.uint32)
    t = (a ^ (a >> 15)) * (a | 1)
    t = (t + (t ^ (t >> 7)) * (t | 61)) ^ t
    return (t ^ (t >> 14)).astype(np.float64) / 2**32


def normals(seed, n):
    u = mulberry32(seed, 2 * n)
    return np.sqrt(-2 * np.log(1 - u[0::2])) * np.cos(2 * np.pi * u[1::2])  # Box–Muller, một số N(0, 1) mỗi cặp


def init(arch, V, H, O, seed, bias=0.0, scale=1.0):
    """V đầu vào one-hot, H đơn vị ẩn, O đầu ra. bias: độ lệch cổng quên b_f (LSTM) hoặc cổng cập nhật b_z (GRU);
    scale: hệ số g của W_h. Thứ tự rút số giống hệt demo (web/src/components/viz/rnn.ts)."""
    G = GATES[arch]
    z = normals(seed, V * G * H + H * G * H + H * O)
    k1, k2 = V * G * H, V * G * H + H * G * H
    p = {
        "Wx": z[:k1].reshape(V, G * H) / np.sqrt(V),
        "Wh": z[k1:k2].reshape(H, G * H) * (scale / np.sqrt(H)),
        "b": np.zeros(G * H),
        "Wy": z[k2:].reshape(H, O) / np.sqrt(H),
        "by": np.zeros(O),
    }
    if arch != "rnn":
        p["b"][H : 2 * H] = bias  # khối cột thứ hai: cổng quên f (LSTM) hoặc cổng cập nhật z (GRU)
    return p


K, H = MEM["K"], MEM["H"]  # 4 ký hiệu A–D, 32 đơn vị ẩn
SYM = "ABCD?"  # ký hiệu số 4 là dấu hỏi
p_rnn = init("rnn", K + 1, H, K, seed=1)
print({k: v.shape for k, v in p_rnn.items()}, "→", vn(sum(v.size for v in p_rnn.values()), 0), "tham số")
print("bốn số nguyên đầu của mulberry32(1):", (mulberry32(1, 4) * 2**32).astype(np.int64).tolist())
assert (mulberry32(1, 4) * 2**32).tolist() == [2693262067, 11749833, 2265367787, 4213581821], (
    "khác với bộ sinh của demo"
)

Phần đo gradient của demo dùng **64 chuỗi cố định**: ký hiệu $\lfloor 4u \rfloor$ với $u$ lấy từ `mulberry32(12345)`,
mỗi hàng 200 ký hiệu, rồi cắt còn $T$ ký hiệu và thêm “?”. Ô dưới dựng lại chúng, rồi cho chuỗi đầu tiên (cắt ở
$T = 8$) đi qua RNN chưa học, từng bước một. Hàm `step()` là một bước; `forward()` chỉ là vòng lặp gọi nó với cùng
bộ trọng số. Biến `c` để dành cho ô nhớ của LSTM (mục 6); RNN chỉ chuyền nó đi nguyên vẹn.

In [ ]:
PROBE = MEM["probe"]  # seed 12345, 64 chuỗi, tối đa 200 ký hiệu
S_PROBE = np.floor(mulberry32(PROBE["seed"], PROBE["n"] * PROBE["maxT"]) * K).astype(int).reshape(PROBE["n"], -1)


def probe(T):
    """64 chuỗi của demo: T ký hiệu rồi dấu “?”; đáp án là ký hiệu đầu."""
    X = np.concatenate([S_PROBE[:, :T], np.full((PROBE["n"], 1), K)], 1)
    return X, X[:, 0].copy()


def step_rnn(p, x, h, c):
    """Một bước: x là chỉ số ký hiệu (một mảng, mỗi chuỗi một số); trả về h mới, c, và những gì lượt ngược cần."""
    h_new = np.tanh(p["Wx"][x] + h @ p["Wh"] + p["b"])  # p["Wx"][x] = x_t W_x với x_t one-hot
    return h_new, c, (h, h_new)


STEP = {"rnn": step_rnn}  # mục 6 thêm "lstm" và "gru"


def forward(arch, p, X, h=None, c=None):
    """X: mảng số nguyên (B, T). Trả về trạng thái ở mọi bước (B, T, H), h và c cuối, và bộ nhớ đệm cho lượt ngược."""
    B, T = X.shape
    Hn = p["Wh"].shape[0]
    h = np.zeros((B, Hn)) if h is None else h
    c = np.zeros((B, Hn)) if c is None else c
    hs, cache = [], []
    for t in range(T):  # trải theo thời gian: cùng p ở mọi bước
        h, c, e = STEP[arch](p, X[:, t], h, c)
        hs.append(h)
        cache.append(e)
    return np.stack(hs, 1), h, c, cache


X8, y8 = probe(8)
Hs, hT, _, _ = forward("rnn", p_rnn, X8[:1])
print("chuỗi:", " ".join(SYM[s] for s in X8[0]), " → đáp án", SYM[y8[0]])
for t in range(X8.shape[1]):
    print(f"bước {t}: đọc {SYM[X8[0, t]]}, h[:4] = {' '.join(f'{vn(v, 3):>6}' for v in Hs[0, t, :4])} …")
P8 = np.exp(hT @ p_rnn["Wy"]) / np.exp(hT @ p_rnn["Wy"]).sum()
print("chưa học, mạng đoán:", " ".join(f"{SYM[k]} {pct(P8[0, k], 0)}" for k in range(K)))

Mỗi bước, $h$ đổi theo ký hiệu vừa đọc, nhưng vẫn mang theo một phần của $h$ cũ qua $h_{t-1}W_h$. Chưa học thì mạng
đoán gần như đều cho cả bốn ký hiệu. Muốn học, ta cần gradient của mất mát theo $W_x$, $W_h$, $b$.

## 4 · Lan truyền ngược theo thời gian (BPTT)

Trải vòng lặp theo thời gian, RNN là một mạng **sâu $T + 1$ lớp** mà mọi lớp dùng chung một bộ trọng số. Lan truyền ngược
trên mạng trải ra ấy gọi là BPTT (*backpropagation through time*). Đặt $\delta_t = \partial L/\partial z_t$ với
$z_t = x_tW_x + h_{t-1}W_h + b$ là giá trị trước tanh. Đi từ bước cuối về bước đầu:

$$\delta_t = \frac{\partial L}{\partial h_t} \odot (1 - h_t^2), \qquad
\frac{\partial L}{\partial h_{t-1}} \mathrel{+}= \delta_t W_h^{\top}, \qquad
\frac{\partial L}{\partial W_h} = \sum_t h_{t-1}^{\top}\,\delta_t$$

Dòng giữa là chỗ quan trọng: mỗi bước lùi, gradient **nhân với $W_h^{\top}$ và với $1 - h_t^2$**. Dòng cuối là hệ quả
của việc dùng chung trọng số: gradient của $W_h$ là **tổng** đóng góp của mọi bước (tương tự cho $W_x$ và $b$).

Hàm `back_rnn()` là một bước lùi; `backward()` gọi nó từ $t = T$ về $t = 0$ và cộng dồn. Tham số `k` cho phép dừng sau
$k$ bước lùi: đó là **BPTT cắt ngắn**, dùng ở mục 10. Nó cũng ghi lại $\lVert \partial L/\partial h_t \rVert$ ở mọi bước
(trung bình trên lô), thứ mà mục 5 sẽ vẽ.

In [ ]:
def back_rnn(p, e, dh, dc):
    """Một bước lùi. Vào: ∂L/∂hₜ (dh). Ra: δₜ = ∂L/∂zₜ, phần ∂L/∂W_h của bước này, ∂L/∂hₜ₋₁, và dc (không dùng)."""
    h_prev, h_new = e
    dz = dh * (1 - h_new**2)  # tanh′(z) = 1 − tanh²(z)
    return dz, h_prev.T @ dz, dz @ p["Wh"].T, dc


BACK = {"rnn": back_rnn}  # mục 6 thêm "lstm" và "gru"


def backward(arch, p, X, cache, dHs, k=None):
    """BPTT từ dHs[:, t] = ∂L/∂hₜ đi qua đầu ra. k: chỉ đi ngược k bước (BPTT cắt ngắn); None: cả chuỗi.
    Trả về gradient của W_x, W_h, b và ‖∂L/∂hₜ‖ (trung bình trên lô) ở mọi bước t."""
    B, T, Hn = dHs.shape
    g = {n: np.zeros_like(p[n]) for n in ("Wx", "Wh", "b")}
    dh, dc = np.zeros((B, Hn)), np.zeros((B, Hn))
    norms = np.zeros(T)
    for t in reversed(range(T)):
        if k is not None and t < T - k:
            break
        dh = dh + dHs[:, t]  # gradient tới hₜ: qua đầu ra ở bước t, cộng phần từ mọi bước sau
        norms[t] = np.sqrt((dh**2).sum(1)).mean()
        dz, dWh, dh, dc = BACK[arch](p, cache[t], dh, dc)
        np.add.at(g["Wx"], X[:, t], dz)  # cùng W ở mọi bước: gradient là tổng qua các bước
        g["Wh"] += dWh
        g["b"] += dz.sum(0)
    return g, norms


def loss_grad(arch, p, X, y, k=None):
    """Entropy chéo của đáp án y, đọc từ trạng thái cuối. Trả về mất mát, độ chính xác, gradient, ‖∂L/∂hₜ‖."""
    Hs, hT, _, cache = forward(arch, p, X)
    B = len(y)
    logits = hT @ p["Wy"] + p["by"]
    P = np.exp(logits - logits.max(1, keepdims=True))
    P /= P.sum(1, keepdims=True)
    loss = -np.log(P[np.arange(B), y] + 1e-300).mean()
    d = P.copy()
    d[np.arange(B), y] -= 1
    d /= B  # ∂L/∂logits = (P − onehot)/B, như Bài 04
    dHs = np.zeros_like(Hs)
    dHs[:, -1] = d @ p["Wy"].T  # chỉ bước cuối có đầu ra
    g, norms = backward(arch, p, X, cache, dHs, k)
    g["Wy"], g["by"] = hT.T @ d, d.sum(0)
    return loss, (P.argmax(1) == y).mean(), g, norms

Viết tay lượt ngược rất dễ sai một chỉ số. Ta kiểm tra bằng **sai phân hữu hạn** (Bài 06): nhích từng tham số một
lượng $\pm 10^{-6}$, đo mất mát thay đổi bao nhiêu, rồi so với gradient của `backward()`. Mạng thử nhỏ (5 ký hiệu,
4 đơn vị ẩn, chuỗi dài 6) để kiểm từng tham số một. Ô in sai số tương đối lớn nhất.

In [ ]:
def gradcheck(arch, eps=1e-6):
    rng = np.random.default_rng(0)
    p = init(arch, 5, 4, 3, seed=1, bias=0.5)
    p["b"] += 0.1 * rng.standard_normal(p["b"].shape)  # độ lệch khác 0 để kiểm cả phần đó
    X, y = rng.integers(0, 5, (3, 6)), rng.integers(0, 3, 3)
    g = loss_grad(arch, p, X, y)[2]
    worst = 0.0
    for name in p:
        for idx in np.ndindex(p[name].shape):
            old = p[name][idx]
            p[name][idx] = old + eps
            lp = loss_grad(arch, p, X, y)[0]
            p[name][idx] = old - eps
            lm = loss_grad(arch, p, X, y)[0]
            p[name][idx] = old
            num = (lp - lm) / (2 * eps)
            worst = max(worst, abs(num - g[name][idx]) / max(1e-8, abs(num) + abs(g[name][idx])))
    return worst


err = gradcheck("rnn")
print(f"RNN: sai số tương đối lớn nhất giữa BPTT và sai phân hữu hạn = {p10(np.log10(err))}")
assert err < 1e-5, "lượt ngược sai"

Sai số cỡ $10^{-9}$, đúng mức làm tròn của sai phân hữu hạn: `backward()` đúng.

## 5 · Gradient đi ngược được bao xa?

Bây giờ đo trên mạng thật của demo: RNN 32 đơn vị ẩn lúc **khởi tạo**, 64 chuỗi của demo, $T = 80$ ký hiệu. Đại lượng
cần nhìn là

$$\frac{\lVert \partial L/\partial h_t \rVert}{\lVert \partial L/\partial h_T \rVert}$$

(trung bình trên 64 chuỗi): phần gradient ở bước cuối (dấu “?”) còn về tới bước $t$. Ký hiệu cần nhớ nằm ở bước 0,
nên nếu tỉ số ở bước 0 rất nhỏ, gradient gần như không cho mạng biết phải sửa gì để nhớ nó.

### 🤔 Dự đoán trước
Với $T = 80$ và hệ số $g = 1$ (trọng số $W_h \sim \mathcal N(0, 1/H)$, khởi tạo “chuẩn”), phần gradient về tới bước 0
vào cỡ nào: gần 1, khoảng 0,1, hay nhỏ hơn $10^{-5}$?

In [ ]:
def profile(arch, p, T):
    """log10 của ‖∂L/∂hₜ‖ / ‖∂L/∂h_T‖, t = 0…T, trên 64 chuỗi của demo (như memory.profile)."""
    X, y = probe(T)
    norms = loss_grad(arch, p, X, y)[3]
    return np.log10(np.maximum(norms / norms[-1], 1e-300))


prof = profile("rnn", init("rnn", K + 1, H, K, seed=1), 80)
print(f"RNN, g = 1, T = 80: bước 0 nhận {p10(prof[0])} gradient của bước cuối")
print(f"trung bình mỗi bước lùi: ×{vn(10 ** (prof[0] / 80), 3)}")
# so với profile mà memory.py lưu cho demo (seed 0, cùng 64 chuỗi): mục 7 kiểm phần còn lại
ref = next(c for c in MEM["checks"] if c["arch"] == "rnn" and c["scale"] == 1.0 and c["T"] == 80)
assert np.abs(prof - np.array(ref["profile"])).max() < 1e-4, "khác với demo"

Nhỏ hơn $10^{-5}$ rất nhiều: chỉ còn khoảng $3{,}7 \cdot 10^{-8}$. Trung bình mỗi bước lùi gradient nhân với 0,808, và
80 thừa số như thế nhân với nhau. Đây đúng là bài toán của Bài 09, chỉ khác là “lớp” ở đây là bước thời gian và **mọi lớp
dùng chung một ma trận** $W_h$.

Hệ số $g$ của $W_h$ quyết định thừa số mỗi bước. Hình trái vẽ tỉ số theo bước $t$ (thang log) với ba giá trị $g$: đường
thẳng trên thang log nghĩa là mỗi bước nhân một thừa số gần như cố định. Hình phải quét $g$ từ 0,25 đến 3 với $T = 50$
và vẽ phần gradient về tới bước 0; đường gạch ngang là mức 1 (giữ nguyên).

In [ ]:
GRAD_LABEL = r"$\Vert\partial L/\partial h_t\Vert \,/\, \Vert\partial L/\partial h_T\Vert$"  # trục y (mathtext)


def mt(label):
    """Nhãn trong hình: b_f, b_z viết thành chỉ số dưới."""
    return label.replace("b_f", "$b_f$").replace("b_z", "$b_z$")


fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.6))
T = 80
for g, color, marker, ls in [(0.5, ORANGE, "s", ":"), (1.0, BLUE, "o", "-"), (2.0, GREEN, "^", "--")]:
    pr = profile("rnn", init("rnn", K + 1, H, K, 1, scale=g), T)
    a.plot(np.arange(T + 1), 10.0**pr, color=color, ls=ls, marker=marker, ms=4, markevery=10, label=f"g = {vn(g, 1)}")
    print(f"g = {vn(g, 1)}: bước 0 nhận {p10(pr[0])}")
a.set_yscale("log")
a.set(
    xlabel="bước t (← gradient đi ngược từ bước cuối)",
    ylabel=GRAD_LABEL,
    title=f"RNN lúc khởi tạo, T = {T}",
)
a.legend(frameon=False, fontsize=9)
gs = np.round(np.arange(0.25, 3.001, 0.05), 2)
g0 = np.array([profile("rnn", init("rnn", K + 1, H, K, 1, scale=g), 50)[0] for g in gs])
print(f"quét {len(gs)} giá trị g từ {vn(gs[0], 2)} đến {vn(gs[-1], 0)}")
b.plot(gs, 10.0**g0, color=BLUE, marker="o", ms=3)
b.axhline(1, color=GREY, ls="--", lw=1)
b.set_yscale("log")
b.set(xlabel="hệ số g của Wₕ", ylabel="phần gradient về tới bước 0", title="T = 50: co hay bùng nổ?")
for ax in (a, b):
    ax.grid(alpha=0.3)
plt.tight_layout()
vn_axes(fig)
plt.show()
g_cross = gs[np.argmax(g0 > 0)]
W1 = init("rnn", K + 1, H, K, 1)["Wh"]
print(
    f"T = 50: g = 2 cho bước 0 nhận {p10(g0[gs == 2.0][0])}; gradient bắt đầu lớn hơn ở bước cuối từ g = {vn(g_cross, 2)}"
)
print(
    f"với g = 1: bán kính phổ của W_h = {vn(np.abs(np.linalg.eigvals(W1)).max(), 2)}, giá trị kỳ dị lớn nhất = {vn(np.linalg.norm(W1, 2), 2)}"
)

Với $g$ nhỏ, gradient co theo hàm mũ; với $g$ lớn, nó **bùng nổ** theo hàm mũ: với $g = 2$, về tới bước 0 lớn gấp 220
lần ở bước cuối khi $T = 50$, và gấp 9.014 lần khi $T = 80$. Chỉ một dải hẹp quanh g = 1,55 giữ được gần nguyên, và vị
trí của dải đó còn đổi theo độ dài chuỗi và theo trọng số trong lúc học. Pascanu, Mikolov & Bengio (2013) chứng minh: nếu giá trị kỳ dị lớn nhất của $W_h$ nhỏ hơn
$1/\gamma$ (với $\gamma = 1$ cho tanh, cận trên của đạo hàm) thì gradient chắc chắn biến mất; muốn bùng nổ thì nó *phải*
lớn hơn $1/\gamma$. Ở $g = 1$, $W_h$ có giá trị kỳ dị lớn nhất 1,89 mà gradient vẫn co: điều kiện bùng nổ là cần, chưa
đủ, vì $1 - h_t^2$ thường nhỏ hơn 1 nhiều.

Gradient bùng nổ có cách chữa đơn giản: **cắt ngưỡng gradient** (Bài 09), như mọi lần huấn luyện trong bài này (tổng
chuẩn tối đa 1). Gradient biến mất thì không chữa bằng cách đó được, vì không thể phóng to một tín hiệu đã chìm. Cần đổi
kiến trúc.

## 6 · LSTM và GRU: những cánh cổng

Một **cổng** là một vectơ sigmoid trong khoảng $(0, 1)$, tính từ $x_t$ và $h_{t-1}$; nhân từng phần tử với nó nghĩa là
“cho qua bao nhiêu”. LSTM (Hochreiter & Schmidhuber 1997; cổng quên của Gers và cộng sự 2000) thêm một **ô nhớ** $c_t$:

$$[\,i_t \;\; f_t \;\; o_t\,] = \sigma(x_t W_x + h_{t-1} W_h + b)_{\,i,f,o}, \qquad \tilde c_t = \tanh(\cdots)_{\,\tilde c}$$

$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde c_t, \qquad h_t = o_t \odot \tanh(c_t)$$

Cổng quên $f_t$ quyết định giữ bao nhiêu ô nhớ cũ, cổng nhập $i_t$ thêm bao nhiêu giá trị đề cử $\tilde c_t$, cổng xuất
$o_t$ cho bao nhiêu ra $h_t$. Bốn khối cột của $W_x$, $W_h$, $b$ (thứ tự $i, f, o, \tilde c$) được tính bằng **một** phép
nhân ma trận. Đường đi của ô nhớ chỉ có phép nhân với cổng và phép cộng:

$$\frac{\partial c_t}{\partial c_{t-1}} = f_t \quad\text{(từng phần tử, chưa kể các đường đi qua cổng)}$$

không có ma trận trọng số, không có tanh. Với $f_t = 1$ đó là một kết nối tắt đồng nhất theo thời gian, như ResNet
(Bài 13). GRU (Cho và cộng sự 2014) gọn hơn: không có ô nhớ riêng, hai cổng $r_t$ (đặt lại) và $z_t$ (cập nhật):

$$\tilde h_t = \tanh\!\big(x_t W_x + (r_t \odot h_{t-1}) W_h + b\big)_{\tilde h}, \qquad h_t = z_t \odot h_{t-1} + (1 - z_t) \odot \tilde h_t$$

theo quy ước của Cho và cộng sự: $z_t$ gần 1 là **giữ** trạng thái cũ (có tài liệu, như PyTorch, áp $r_t$ sau phép nhân
với $W_h$; có tài liệu đổi vai $z_t$ và $1 - z_t$). Trong code, cổng cập nhật tên `u` để khỏi nhầm với $z$ trước kích hoạt.
Độ lệch của cổng ($b_f$ cho LSTM, $b_z$ cho GRU) là núm khởi tạo mà mục 7 sẽ vặn.

In [ ]:
def step_lstm(p, x, h, c):
    Hn = p["Wh"].shape[0]
    z = p["Wx"][x] + h @ p["Wh"] + p["b"]  # một phép nhân cho cả bốn khối: i, f, o, c̃
    i, f, o = sig(z[:, :Hn]), sig(z[:, Hn : 2 * Hn]), sig(z[:, 2 * Hn : 3 * Hn])
    g = np.tanh(z[:, 3 * Hn :])  # giá trị đề cử c̃
    c_new = f * c + i * g  # băng chuyền: chỉ nhân với cổng và cộng
    tc = np.tanh(c_new)
    return o * tc, c_new, (h, c, i, f, o, g, tc)


def step_gru(p, x, h, c):
    Hn = p["Wh"].shape[0]
    a = p["Wx"][x] + p["b"]
    zr = a[:, : 2 * Hn] + h @ p["Wh"][:, : 2 * Hn]
    r, u = sig(zr[:, :Hn]), sig(zr[:, Hn:])  # cổng đặt lại r, cổng cập nhật z (tên u)
    rh = r * h
    n = np.tanh(a[:, 2 * Hn :] + rh @ p["Wh"][:, 2 * Hn :])  # giá trị đề cử h̃
    return u * h + (1 - u) * n, c, (h, r, u, rh, n)


def back_lstm(p, e, dh, dc):
    h, c, i, f, o, g, tc = e
    dcn = dc + dh * o * (1 - tc**2)  # gradient tới cₜ: từ hₜ và từ cₜ₊₁
    dz = np.concatenate([dcn * g * i * (1 - i), dcn * c * f * (1 - f), dh * tc * o * (1 - o), dcn * i * (1 - g**2)], 1)
    return dz, h.T @ dz, dz @ p["Wh"].T, dcn * f  # ∂cₜ/∂cₜ₋₁ = fₜ: dòng cuối của băng chuyền


def back_gru(p, e, dh, dc):
    Hn = p["Wh"].shape[0]
    h, r, u, rh, n = e
    dzn = dh * (1 - u) * (1 - n**2)
    drh = dzn @ p["Wh"][:, 2 * Hn :].T
    dzr = np.concatenate([drh * h * r * (1 - r), dh * (h - n) * u * (1 - u)], 1)
    dWh = np.concatenate([h.T @ dzr, rh.T @ dzn], 1)
    dh_prev = dh * u + drh * r + dzr @ p["Wh"][:, : 2 * Hn].T  # đường thẳng u ⊙ dh, rồi qua r và qua các cổng
    return np.concatenate([dzr, dzn], 1), dWh, dh_prev, dc


STEP.update(lstm=step_lstm, gru=step_gru)
BACK.update(lstm=back_lstm, gru=back_gru)
for arch in ("lstm", "gru"):
    err = gradcheck(arch)
    print(f"{arch.upper()}: sai số tương đối lớn nhất giữa BPTT và sai phân hữu hạn = {p10(np.log10(err))}")
    assert err < 1e-4, "lượt ngược sai"
for arch in ("rnn", "gru", "lstm"):
    n = sum(v.size for v in init(arch, K + 1, H, K, 1).values())
    print(f"{arch.upper():>4}, H = {H}: {vn(n, 0)} tham số")

Cả ba lượt ngược đều khớp với sai phân hữu hạn. Với cùng $H = 32$, GRU có gấp ba và LSTM gấp bốn số trọng số hồi tiếp
của RNN, vì mỗi cổng cần một khối cột riêng.

## 7 · Cổng giữ gradient, nhưng chỉ khi được khởi tạo để giữ

Đo lại tỉ số gradient của mục 5 cho LSTM và GRU, cũng lúc khởi tạo, cùng 64 chuỗi, $T = 80$. Mặc định mọi độ lệch bằng 0,
nên lúc đầu các cổng quanh $\sigma(0) = 0{,}5$.

### 🤔 Dự đoán trước
LSTM có băng chuyền ô nhớ mà gradient đi qua không cần nhân ma trận. Với độ lệch cổng quên $b_f = 0$, phần gradient về
tới bước 0 của LSTM **lớn hơn hay nhỏ hơn** của RNN thường ($3{,}7 \cdot 10^{-8}$)?

In [ ]:
def keep_gate(arch, p, T):
    """Cổng “giữ” trung bình ở từng bước (LSTM: fₜ, GRU: zₜ) trên 64 chuỗi của demo, và log10 tích của nó từ t+1 tới T:
    phần gradient chỉ đi theo đường tắt (băng chuyền của LSTM, đường zₜ ⊙ hₜ₋₁ của GRU)."""
    cache = forward(arch, p, probe(T)[0])[3]
    gate = np.array([e[3].mean() if arch == "lstm" else e[2].mean() for e in cache])
    return gate, np.r_[np.cumsum(np.log10(gate[1:])[::-1])[::-1], 0.0]


T = 80
CFG7 = [  # (kiến trúc, độ lệch cổng, nhãn, màu, dấu, nét)
    ("rnn", 0.0, "RNN, g = 1", ORANGE, "s", ":"),
    ("lstm", 0.0, "LSTM, b_f = 0", BLUE, "o", "-"),
    ("lstm", 3.0, "LSTM, b_f = 3", BLUE, "D", "--"),
    ("gru", 0.0, "GRU, b_z = 0", GREEN, "^", "-"),
    ("gru", 3.0, "GRU, b_z = 3", GREEN, "v", "--"),
]
prof7 = {}
for arch, bias, label, *_ in CFG7:
    p = init(arch, K + 1, H, K, 1, bias=bias)
    prof7[label] = profile(arch, p, T)
    line = f"{label:>14}: bước 0 nhận {p10(prof7[label][0]):>10}"
    if arch != "rnn":
        gate, keep = keep_gate(arch, p, T)
        line += f"   cổng trung bình {vn(gate[1:].mean(), 3)}, riêng đường tắt: {p10(keep[0])}"
    print(line)
ratio = prof7["RNN, g = 1"][0] - prof7["LSTM, b_f = 0"][0]
print(f"RNN thường giữ được nhiều hơn LSTM với b_f = 0: gấp {p10(ratio)} lần")
assert prof7["LSTM, b_f = 0"][0] < prof7["RNN, g = 1"][0] < prof7["LSTM, b_f = 3"][0]

**Nhỏ hơn**, khoảng $2 \cdot 10^5$ lần: LSTM với $b_f = 0$ chỉ đưa về bước 0 khoảng $1{,}9 \cdot 10^{-13}$ gradient.
Băng chuyền chỉ giữ gradient khi cổng quên gần 1. Lúc khởi tạo $f_t \approx 0{,}5$, nên riêng trên băng chuyền gradient mất một nửa
mỗi bước: sau 80 bước còn $0{,}5^{80} \approx 10^{-24}$. Phần lớn gradient còn lại đi qua $h$ và các cổng, tức là qua
$W_h$ như RNN, và mỗi đường lại có thêm một thừa số sigmoid nhỏ hơn 1. LSTM **không tự hết** gradient biến mất; nó chỉ
cho mạng một đường mà gradient *có thể* đi nguyên vẹn. (Bài viết nổi tiếng của colah gọi việc nhớ lâu là “hành vi gần như
mặc định” của LSTM; điều đó chỉ đúng khi cổng quên mở gần hết.)

Cộng $b_f = 3$ thì $f_t \approx \sigma(3) \approx 0{,}95$ ngay từ đầu, và gradient về tới bước 0 bằng 0,2 lần ở bước
cuối. GRU giống hệt: cổng cập nhật $z_t$ đóng vai “giữ”. Đó là lý do Gers và cộng sự (2000) rồi Jozefowicz, Zaremba & Sutskever (2015) khuyên
cộng 1 (hoặc hơn) vào độ lệch cổng quên; Keras đặt mặc định `unit_forget_bias=True`, tức $b_f = 1$.

Hình dưới vẽ tỉ số gradient theo bước (thang log). Các đường mảnh chấm chấm cùng màu là riêng phần đi theo đường tắt:
tích các cổng $f_t$ (LSTM) hoặc $z_t$ (GRU) từ bước $t + 1$ tới bước cuối.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for arch, bias, label, color, marker, ls in CFG7:
    pr = prof7[label]
    ax.plot(np.arange(T + 1), 10.0**pr, color=color, ls=ls, marker=marker, ms=4, markevery=10, lw=1.8, label=mt(label))
    if arch != "rnn":
        ax.plot(
            np.arange(T + 1),
            10.0 ** keep_gate(arch, init(arch, K + 1, H, K, 1, bias=bias), T)[1],
            color=color,
            ls=":",
            lw=1,
        )
ax.set_yscale("log")
ax.set_ylim(1e-26, 3)
ax.set(xlabel="bước t (← gradient đi ngược từ bước cuối)", ylabel=GRAD_LABEL)
ax.set_title("Lúc khởi tạo, T = 80: đường đậm là tổng gradient, chấm mảnh là riêng đường tắt", fontsize=11)
ax.legend(frameon=False, fontsize=9, ncol=2, loc="lower right")
ax.grid(alpha=0.3)
vn_axes(fig)
plt.show()

Các đường đậm của LSTM và GRU với độ lệch 0 rơi còn nhanh hơn RNN (cam), và đường chấm mảnh của chúng (riêng băng
chuyền) rơi thẳng xuống $10^{-24}$. Với độ lệch 3, cả hai gần như nằm ngang.

Demo tính đúng những đường này trong trình duyệt. Ô dưới so toàn bộ với tệp của demo: các profile tham chiếu (RNN với
$g = 1$ và 2, GRU và LSTM với độ lệch 0 và 3, $T$ tới 200), và profile lúc khởi tạo của cả 90 lần huấn luyện ở mục 8
(seed $s$ bắt đầu từ `mulberry32(s + 1)`).

In [ ]:
worst = 0.0
for c in MEM["checks"]:
    pr = profile(c["arch"], init(c["arch"], K + 1, H, K, 1, c.get("bias", 0.0), c.get("scale", 1.0)), c["T"])
    worst = max(worst, np.abs(pr - np.array(c["profile"])).max())
CONFIGS = {c["id"]: c for c in MEM["configs"]}
for r in MEM["runs"]:
    c = CONFIGS[r["id"]]
    for s, sd in zip(MEM["seeds"], r["seeds"]):
        pr = profile(c["arch"], init(c["arch"], K + 1, H, K, s + 1, bias=c["bias"]), r["T"])
        worst = max(worst, np.abs(pr - np.array(sd["before"])).max())  # tệp lưu 2 chữ số thập phân
n_prof = len(MEM["checks"]) + len(MEM["runs"]) * len(MEM["seeds"])
print(f"{n_prof} profile gradient, chênh lệch lớn nhất với tệp của demo (log10): {vn(worst, 4)}")
assert worst < 0.006, "khác với demo: hãy kiểm tra lại trọng số khởi tạo"

## 8 · Huấn luyện thật: nhớ ký hiệu đầu

Gradient lúc khởi tạo mới chỉ là dự báo. Giờ huấn luyện thật, đúng công thức của demo (`reel/rnn/memory.py`):
- mỗi bước một lô 64 chuỗi **mới**, rút từ `default_rng(seed + 99)`; tập kiểm tra là 1.000 chuỗi cố định từ
  `default_rng(12345)`;
- Adam (Bài 07) với $\eta = 0{,}003$, cắt ngưỡng tổng chuẩn gradient ở 1 (Bài 09);
- seed 0 bắt đầu từ đúng trọng số của mục 3–7.

Demo chạy 5.000 bước; ở đây ta chạy 300 bước với $T = 20$ cho ba mạng: RNN, LSTM với $b_f = 0$ và LSTM với
$b_f = 1$ (mặc định của Keras), rồi so với điểm đo đầu tiên của demo (bước 250). Ô in độ chính xác kiểm tra mỗi 50 bước.

In [ ]:
LR, CLIP, BATCH = MEM["lr"], MEM["clip"], MEM["batch"]
print(f"demo: {vn(MEM['steps'], 0)} bước, lô {BATCH}, η = {vn(LR, 3)}, tập kiểm tra {vn(MEM['nTest'], 0)} chuỗi")


def batch(rng, T, n):
    X = rng.integers(0, K, (n, T + 1))
    X[:, -1] = K  # dấu “?” ở cuối
    return X, X[:, 0].copy()


def accuracy(arch, p, X, y):
    logits = forward(arch, p, X)[1] @ p["Wy"] + p["by"]
    return (logits.argmax(1) == y).mean()


def train_memory(arch, bias, T, seed=0, steps=300, k=None, every=50, scale=1.0):
    p = init(arch, K + 1, H, K, seed + 1, bias=bias, scale=scale)
    m = {n: np.zeros_like(v) for n, v in p.items()}
    s2 = {n: np.zeros_like(v) for n, v in p.items()}
    Xt, yt = batch(np.random.default_rng(12345), T, MEM["nTest"])
    rng = np.random.default_rng(seed + 99)
    curve = []
    for s in range(1, steps + 1):
        X, y = batch(rng, T, BATCH)
        g = loss_grad(arch, p, X, y, k)[2]
        gn = np.sqrt(sum((v**2).sum() for v in g.values()))
        if gn > CLIP:  # cắt ngưỡng tổng chuẩn gradient
            g = {n: v * (CLIP / gn) for n, v in g.items()}
        for n in p:  # Adam
            m[n] = 0.9 * m[n] + 0.1 * g[n]
            s2[n] = 0.999 * s2[n] + 0.001 * g[n] ** 2
            p[n] -= LR * (m[n] / (1 - 0.9**s)) / (np.sqrt(s2[n] / (1 - 0.999**s)) + 1e-8)
        if s % every == 0:
            curve.append(accuracy(arch, p, Xt, yt))
    return p, np.array(curve)


live = {}
for cid, arch, bias in [("rnn", "rnn", 0.0), ("lstm0", "lstm", 0.0), ("lstm1", "lstm", 1.0)]:
    live[cid] = train_memory(arch, bias, T=20)[1]
    ref = next(r for r in MEM["runs"] if r["id"] == cid and r["T"] == 20)["seeds"][0]["curve"][0]  # bước 250
    print(f"{CONFIGS[cid]['label']:>14}: " + " ".join(pct(a, 0) for a in live[cid]))
    print(f"{'':>14}  bước 250: {pct(live[cid][4], 1)} (demo {pct(ref, 1)})")
    # cùng trọng số đầu, cùng dòng số ngẫu nhiên: trùng demo (cho phép lệch nhỏ vì phép làm tròn khác máy)
    assert abs(live[cid][4] - ref) < 0.05, "khác với demo"

Đúng như gradient lúc khởi tạo dự báo. RNN thường và LSTM với $b_f = 1$ nhớ được hoàn toàn chỉ sau vài trăm bước, còn
LSTM với $b_f = 0$ kẹt ở mức đoán mò 25%: tín hiệu “hãy giữ ký hiệu đầu” về tới bước 0 quá yếu để mạng học được. Ba đường
trùng với demo vì cùng trọng số đầu và cùng các lô.

## 9 · Kết quả quét: 6 mạng × 5 độ dài × 3 seed

Demo chạy công thức trên cho 6 cấu hình (RNN; GRU với $b_z$ = 0 hoặc 3; LSTM với $b_f$ = 0, 1 hoặc 3), $T$ từ 10 đến
160, mỗi ô 3 seed, 5.000 bước. Hình dưới đếm số seed “nhớ được” (đúng ≥ 99% trên 1.000 chuỗi kiểm tra) trong mỗi ô:
màu đậm và con số ghi trong ô cùng cho biết điều đó.

In [ ]:
Ts, cids = MEM["Ts"], [c["id"] for c in MEM["configs"]]
RUN = {(r["id"], r["T"]): r["seeds"] for r in MEM["runs"]}
solved = np.array([[sum(s["acc"] >= 0.99 for s in RUN[(c, T)]) for T in Ts] for c in cids])
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.imshow(solved, cmap=LinearSegmentedColormap.from_list("trang_xanh", ["#f2f2f2", BLUE]), vmin=0, vmax=3)
for i, c in enumerate(cids):
    for j, T in enumerate(Ts):
        acc = [s["acc"] for s in RUN[(c, T)]]
        ax.text(
            j,
            i,
            f"{solved[i, j]}/3",
            ha="center",
            va="center",
            fontsize=10,
            color="white" if solved[i, j] >= 2 else "black",
        )
        print(f"{CONFIGS[c]['label']:>14}, T = {T:>3}: " + " · ".join(pct(a, 1) for a in acc))
ax.set_yticks(range(len(cids)), [mt(CONFIGS[c]["label"]) for c in cids])
ax.set_xticks(range(len(Ts)), [str(T) for T in Ts])
ax.set(xlabel="độ dài chuỗi T", title="Số seed nhớ được ký hiệu đầu (≥ 99%), 5.000 bước")
ax.spines[:].set_visible(False)
vn_axes(fig)
plt.show()

- **RNN thường** nhớ được tới $T = 20$ (2/3 seed), rồi rơi về mức đoán mò từ $T = 40$.
- **LSTM với $b_f = 0$ còn kém hơn RNN**: chỉ nhớ được $T = 10$. Với $b_f = 1$ nó ngang RNN, GRU với $b_z = 0$ cũng vậy.
- **$b_f = 3$ hoặc $b_z = 3$** nhớ được tới $T = 160$ ở 29 trong 30 lần chạy. Khác biệt lớn nhất trong hình không phải
  giữa các kiến trúc, mà giữa hai cách khởi tạo của cùng một kiến trúc.

“Không nhớ được” ở đây nghĩa là trong 5.000 bước với $\eta = 0{,}003$: huấn luyện lâu hơn hoặc chọn $\eta$ khác có thể
đổi vài ô. Tệp của demo cũng lưu profile gradient **sau** huấn luyện (seed 0). Ô dưới in phần gradient về tới bước 0,
trước → sau, với $T = 40$ và $T = 160$.

In [ ]:
for c in cids:
    row = []
    for T in (40, 160):
        s0 = RUN[(c, T)][0]
        row.append(f"T = {T:>3}: {p10(s0['before'][0]):>10} → {p10(s0['after'][0]):>10} ({pct(s0['acc'], 0)})")
    print(f"{CONFIGS[c]['label']:>14}   " + "   ".join(row))

Mạng nào học được thì sau huấn luyện gradient về tới bước 0 vẫn lớn (GRU với $b_z = 3$ còn tăng lên 1,1 và 2,8): mạng đã
chỉnh cổng để giữ. Mạng nào thất bại thì gradient còn **nhỏ hơn** lúc đầu, có khi tới $10^{-44}$: không nhận được tín hiệu
từ bước 0, mạng chỉ học những gì có ở các bước gần, và đường về quá khứ càng mờ đi. Đây là một vòng luẩn quẩn: muốn học
cách nhớ thì gradient phải về được tới chỗ cần nhớ ngay từ đầu.

## 10 · BPTT cắt ngắn

Với chuỗi rất dài (một cuốn sách, một giờ âm thanh), lan truyền ngược qua toàn bộ chuỗi tốn bộ nhớ tỉ lệ với độ dài.
Thực tế người ta dùng **BPTT cắt ngắn**: trạng thái vẫn chuyển tiếp qua cả chuỗi, nhưng gradient chỉ đi ngược $k$ bước.
Tham số `k` của `backward()` làm đúng điều đó.

### 🤔 Dự đoán trước
RNN ở mục 8 học được bài $T = 20$ với BPTT đầy đủ. Nếu gradient chỉ đi ngược $k = 10$ bước, sau 300 bước huấn luyện nó
đúng bao nhiêu phần trăm: vẫn gần 100%, khoảng một nửa, hay quanh 25%?

In [ ]:
rnn_k10 = train_memory("rnn", 0.0, T=20, k=10)[1]
print("RNN, T = 20, k = 10: " + " ".join(pct(a, 0) for a in rnn_k10))
print("RNN, T = 20, đầy đủ: " + " ".join(pct(a, 0) for a in live["rnn"]))
assert rnn_k10.max() < 0.35 < live["rnn"][-1]
lstm3_k10 = train_memory("lstm", 3.0, T=20, k=10)[1]
print("LSTM, b_f = 3, T = 20, k = 10: " + " ".join(pct(a, 0) for a in lstm3_k10))
assert 0.4 < lstm3_k10[-1] < 0.95

Quanh 25%: đoán mò. Với $k = 10$, gradient không bao giờ về tới bước 0, nên mạng **không có cách nào** biết rằng ký hiệu
đầu quan trọng. BPTT cắt ngắn đặt một giới hạn cứng: mạng không *học* được phụ thuộc dài hơn $k$ bước.

Nhưng LSTM với $b_f = 3$, cũng chỉ $k = 10$, lại lên được khoảng 78%. Vì sao? Với $f_t \approx 0{,}95$ ngay từ đầu, ô nhớ
của nó *sẵn* giữ một phần ký hiệu đầu tới cuối chuỗi, dù chưa học gì. Lớp đọc ra $W_y$ chỉ cần gradient ở bước cuối, nên
nó học khai thác được phần trí nhớ có sẵn ấy. Cắt ngắn chặn việc học *cách nhớ*, không chặn việc dùng trí nhớ mà trạng
thái tình cờ mang theo. (Karpathy, 2015, huấn luyện mô hình ký tự trên tiểu luận của Paul Graham với BPTT cắt ở 100 ký tự.)

## 11 · Mô hình ngôn ngữ mức ký tự trên Truyện Kiều

Bài toán thật: đọc Truyện Kiều **từng ký tự** và đoán ký tự tiếp theo. Đầu ra ở *mọi* bước, không chỉ bước cuối, và
mất mát là entropy chéo trung bình. Demo (`reel/rnn/kieu.py`) đã huấn luyện ba loại mạng, $H = 128$, mỗi loại 3 seed:
Adam, 4.000 bước, mỗi bước 32 đoạn 100 ký tự, cắt ngưỡng gradient 1, LSTM khởi tạo $b_f = 1$; $\eta$ chọn trong
{0,001; 0,002; 0,004} theo tập kiểm định, và giữ điểm lưu có mất mát kiểm định thấp nhất. Việc đó mất hàng chục phút trên
một máy nhiều nhân, nên ta nạp **trọng số của seed 0** mà demo tải về. Chúng được lưu dạng int8: mỗi cột của ma trận một hệ số tỉ
lệ $s$, và $W = q \cdot s$ với $q \in \{-127, \dots, 127\}$. Mọi con số của demo đều tính từ chính các trọng số đã làm
tròn này.

In [ ]:
def dequant(d):
    """int8 (base64) với một hệ số tỉ lệ mỗi cột → ma trận số thực, đúng như demo giải nén."""
    q = np.frombuffer(b64decode(d["q"]), np.int8).reshape(d["rows"], d["cols"])
    return q * np.array(d["s"])


LM = {}
for arch, d in KIEU["ship"].items():
    LM[arch] = {k: dequant(d[k]) for k in ("Wx", "Wh", "Wy")} | {k: np.array(d[k], float) for k in ("b", "by")}
MODELS = {(m["arch"], m["H"]): m for m in KIEU["models"]}
print(
    "η thử:",
    " · ".join(vn(lr, 3) for lr in KIEU["lrs"]),
    f"· {vn(KIEU['steps'], 0)} bước, lô {KIEU['batch']} × {KIEU['window']} ký tự",
)
for arch, p in LM.items():
    n = sum(v.size for v in p.values())
    print(f"{arch.upper():>4}, H = 128: {vn(n, 0):>7} tham số, η chọn được = {vn(MODELS[(arch, 128)]['lr'], 3)}")
    assert n == MODELS[(arch, 128)]["params"]

Cho ba mạng đọc bốn câu cuối của tập huấn luyện, rồi so các logit của ký tự tiếp theo với số mà demo
lưu. Mạng đã học thì ổn định, nên chênh lệch chỉ là phép làm tròn của máy. (Một RNN chưa học có thể *hỗn loạn*: sai số
làm tròn bị khuếch đại theo hàm mũ qua từng bước, và phép so này sẽ không đứng được.)

In [ ]:
def run_text(arch, p, text, h=None, c=None):
    """Đọc từng ký tự của text; trả về h và c cuối."""
    h = np.zeros((1, 128)) if h is None else h
    c = np.zeros((1, 128)) if c is None else c
    for ch in text:
        h, c, _ = STEP[arch](p, np.array([IX[ch]]), h, c)
    return h, c


PROMPT = KIEU["prompt"]
print(PROMPT)
for arch, p in LM.items():
    h, _ = run_text(arch, p, PROMPT)
    logits = (h @ p["Wy"] + p["by"])[0]
    err = np.abs(logits - np.array(KIEU["ship"][arch]["check"])).max()
    top = np.argsort(-logits)[:3]
    P = np.exp(logits - logits.max()) / np.exp(logits - logits.max()).sum()
    print(
        f"{arch.upper():>4}: ký tự đầu câu mới, 3 lựa chọn cao nhất: "
        + ", ".join(f"“{CHARS[i]}” {pct(P[i], 1)}" for i in top)
    )
    print(f"      chênh lệch logit lớn nhất so với demo: {p10(np.log10(err))}")
    assert err < 1e-6, "khác với demo"

Logit khớp tới $10^{-8}$: đây đúng là ba mạng của demo. Cả ba đều đoán câu mới bắt đầu bằng một chữ hoa, đúng thói quen
của văn bản.

## 12 · Mạng đoán ký tự tiếp theo giỏi tới đâu?

Đo bằng **bit trên ký tự** trên tập kiểm định: $-\log_2 p(\text{ký tự đúng})$ trung bình, đúng thước đo của demo. Đoán đều
trên 132 ký tự tốn $\log_2 132 \approx 7{,}04$ bit; chỉ dùng tần suất ký tự (không nhớ gì) tốn 5,18 bit. Mạng đọc
512 ký tự cuối của tập huấn luyện để “khởi động” trạng thái, rồi đọc liền một mạch 254 câu kiểm định, mang trạng thái
theo suốt.

Cùng một lượt đọc, ta giữ lại trạng thái và logit ở mọi ký tự: các mục 14 và 15 dùng lại chúng.

In [ ]:
def read_val(arch, p):
    """Khởi động bằng 512 ký tự huấn luyện, rồi đọc tập kiểm định từng ký tự. Trả về, sau mỗi ký tự kiểm định:
    logit của ký tự kế tiếp, trạng thái h, và ô nhớ c (LSTM)."""
    h, c = run_text(arch, p, "".join(CHARS[i] for i in TRAIN[-WARM:]))
    logits, hs, cs = [], [], []
    for x in VAL:
        h, c, _ = STEP[arch](p, np.array([x]), h, c)
        logits.append((h @ p["Wy"] + p["by"])[0])
        hs.append(h[0])
        cs.append(c[0])
    return np.array(logits), np.array(hs), np.array(cs)


def log_softmax(z):
    z = z - z.max(-1, keepdims=True)
    return z - np.log(np.exp(z).sum(-1, keepdims=True))


counts = np.bincount(TRAIN, minlength=V) + 1  # tần suất ký tự trong tập huấn luyện (+1 để không có xác suất 0)
unigram = -np.log2(counts / counts.sum())[VAL[1:]].mean()
print(f"đoán đều: {vn(np.log2(V), 2)} bit/ký tự · chỉ dùng tần suất ký tự: {vn(unigram, 2)} bit/ký tự")
READ = {arch: read_val(arch, p) for arch, p in LM.items()}
for arch, (lg, _, _) in READ.items():
    bits = -log_softmax(lg[:-1])[np.arange(len(VAL) - 1), VAL[1:]].mean() / np.log(2)
    ref = MODELS[(arch, 128)]["seeds"][0]["bits"]
    print(f"{arch.upper():>4}: {vn(bits, 4)} bit/ký tự trên tập kiểm định (demo: {vn(ref, 4)})")
    assert abs(bits - ref) < 1e-3, "khác với demo"
    steps = [sd["step"] for sd in MODELS[(arch, 128)]["seeds"]]
    last = MODELS[(arch, 128)]["seeds"][0]["curve"][-1]  # [bit/ký tự huấn luyện, kiểm định] ở bước 4.000
    print(f"      điểm lưu tốt nhất của 3 seed ở bước {', '.join(vn(v, 0) for v in steps)}", end="")
    print(f"; ở bước 4.000 (seed 0): huấn luyện {vn(last[0], 2)}, kiểm định {vn(last[1], 2)}")

LSTM dự đoán tốt nhất, rồi GRU, rồi RNN; cả ba đều tốt hơn hẳn mức “chỉ biết tần suất ký tự”. Truyện Kiều chỉ có
104.204 ký tự, nên mô hình nhỏ và học nhanh: điểm lưu tốt nhất theo tập kiểm định đến sớm, chưa tới nửa chặng
4.000 bước (ô trên in bước của từng seed). Sau đó mạng bắt đầu **học thuộc** tập huấn luyện (Bài 08): tới bước 4.000,
bit/ký tự trên tập huấn luyện vẫn giảm, còn trên tập kiểm định thì tăng trở lại.

## 13 · Sinh thơ từng ký tự

Sinh văn bản là đọc ngược lại: lấy phân phối của ký tự kế tiếp, **rút một ký tự** theo phân phối đó, cho mạng đọc chính
ký tự ấy, rồi lặp. **Nhiệt độ** $\tau$ chia các logit trước softmax, $p \propto \exp(z/\tau)$: $\tau < 1$ làm phân phối nhọn
hơn (an toàn, dễ lặp), $\tau > 1$ làm nó phẳng hơn (đa dạng, nhiều lỗi chính tả). Demo dùng $\tau = 0{,}8$, bắt đầu
từ bốn câu cuối của tập huấn luyện và rút số bằng `default_rng(0)` cho seed 0.

Để chấm thơ, ta cần luật **lục bát**: một câu 6 âm tiết rồi một câu 8 âm tiết, xen kẽ; âm tiết 2 thanh bằng (ngang,
huyền), âm tiết 4 thanh trắc (sắc, hỏi, ngã, nặng), âm tiết 6 bằng, và câu 8 thêm âm tiết 8 bằng. Thanh điệu đọc được
từ dấu: tách chữ theo chuẩn Unicode NFD thì dấu sắc, hỏi, ngã, nặng thành các ký tự kết hợp riêng. (Ta không kiểm tra vần.)

In [ ]:
TRAC = {"́", "̉", "̃", "̣"}  # dấu sắc, hỏi, ngã, nặng sau khi tách NFD; ngang, huyền là bằng


def syllables(line):
    return re.sub(r"[^\w\s]", " ", line).split()  # bỏ dấu câu, tách theo khoảng trắng


def trac(s):
    return any(ch in TRAC for ch in unicodedata.normalize("NFD", s.lower()))


def tone_ok(w):
    """Âm tiết 2 bằng, 4 trắc, 6 bằng (và 8 bằng ở câu 8)."""
    return len(w) in (6, 8) and not trac(w[1]) and trac(w[3]) and not trac(w[5]) and (len(w) == 6 or not trac(w[7]))


def tone_text(w):
    return " ".join("T" if trac(w[k]) else "B" for k in (1, 3, 5, 7) if k < len(w)) if len(w) in (6, 8) else "–"


def stats(lines):
    """Tỉ lệ câu 6 hoặc 8 âm tiết, cặp câu liền nhau xen kẽ 6↔8, câu đúng luật bằng trắc, câu chép nguyên văn."""
    n = [len(syllables(ln)) for ln in lines]
    p6, p8 = np.mean([k == 6 for k in n]), np.mean([k == 8 for k in n])
    return {
        "n68": np.mean([k in (6, 8) for k in n]),
        "alt": np.mean([(a, b) in ((6, 8), (8, 6)) for a, b in itertools.pairwise(n)]),
        "chance": 2 * p6 * p8,  # xen kẽ nhờ may rủi nếu độ dài mỗi câu chọn độc lập
        "tone": np.mean([tone_ok(syllables(ln)) for ln, k in zip(lines, n) if k in (6, 8)]),
        "copied": np.mean([ln in CORPUS for ln in lines]),
    }


CORPUS = set(LINES)
S = list(itertools.chain.from_iterable(syllables(ln) for ln in LINES))  # âm tiết của cả bài, xáo trộn như demo
random.Random(0).shuffle(S)
it = iter(S)
shuffled = [" ".join(next(it) for _ in range(6 if i % 2 == 0 else 8)) for i in range(len(LINES))]
real, shuf = stats(VAL_LINES), stats(shuffled)
print(
    f"254 câu kiểm định thật: {pct(real['n68'])} câu 6/8 âm tiết, xen kẽ {pct(real['alt'])}, đúng bằng trắc {pct(real['tone'])}"
)
print(f"âm tiết của cả bài xáo trộn, cắt lại thành câu 6/8: đúng bằng trắc {pct(shuf['tone'])}")
assert round(real["tone"], 4) == KIEU["real"]["tone"] and round(shuf["tone"], 4) == KIEU["shuffled"]["tone"]

Thơ thật của tập kiểm định: mọi câu đều 6 hoặc 8 âm tiết, xen kẽ 100%, và gần như mọi câu đúng luật bằng trắc. Xáo trộn
âm tiết của cả bài rồi cắt lại thành câu 6/8 thì chỉ khoảng một câu trong mười đúng luật: đó là mức may rủi của luật thanh.

Giờ cho mỗi mạng sinh 20.000 ký tự (khoảng 600 câu) đúng như demo, rồi chấm. Ô in 6 câu đầu của mỗi mạng, kèm số âm tiết
và thanh ở các vị trí 2‑4‑6(‑8).

In [ ]:
def generate(arch, p, prompt, n, temp, rng):
    h, c = run_text(arch, p, prompt)
    out = []
    for _ in range(n):
        z = (h @ p["Wy"] + p["by"])[0] / temp  # chia logit cho nhiệt độ
        pr = np.exp(z - z.max())
        x = rng.choice(V, p=pr / pr.sum())  # rút một ký tự theo phân phối
        out.append(CHARS[x])
        h, c, _ = STEP[arch](p, np.array([x]), h, c)  # mạng đọc chính ký tự vừa sinh
    return "".join(out)


TEMP, N_SAMPLE = KIEU["temp"], KIEU["nSample"]
print(f"mỗi mạng sinh {vn(N_SAMPLE, 0)} ký tự ở nhiệt độ {vn(TEMP, 1)}")
GEN = {}
for arch, p in LM.items():
    text = generate(arch, p, PROMPT, N_SAMPLE, TEMP, np.random.default_rng(0))
    GEN[arch] = text.split("\n")[:-1]  # câu cuối chưa xong
    print(f"── {arch.upper()} (nhiệt độ {vn(TEMP, 1)}, seed 0)")
    for ln in GEN[arch][:6]:
        w = syllables(ln)
        print(f"  {ln:<44} {len(w)} âm tiết  {tone_text(w)}")
    assert GEN[arch][:8] == MODELS[(arch, 128)]["seeds"][0]["sample"], "khác với demo"

Trông giống tiếng Việt, đúng chính tả phần lớn, thường có dấu phẩy cuối câu 6 và dấu chấm cuối câu 8 như bản gốc, nhưng không
mang nghĩa. Tám câu đầu của mỗi mạng trùng nguyên văn với mẫu mà demo lưu. Chấm cả 20.000 ký tự:

In [ ]:
print(
    f"{'':>5} {'câu':>4} {'6/8 âm tiết':>12} {'xen kẽ 6↔8':>11} {'may rủi 2·p₆·p₈':>16} {'bằng trắc':>10} {'chép':>6}"
)
for arch in LM:
    st = stats(GEN[arch])
    ref = MODELS[(arch, 128)]["seeds"][0]["gen"]
    print(
        f"{arch.upper():>5} {len(GEN[arch]):>4} {pct(st['n68'], 1):>12} {pct(st['alt'], 1):>11} {pct(st['chance'], 1):>16}"
        f" {pct(st['tone'], 1):>10} {pct(st['copied'], 1):>6}"
    )
    for key in ("n68", "alt", "tone", "copied"):  # cho phép lệch nhỏ: một lần rút khác đi sẽ đổi phần sau của mẫu
        assert abs(st[key] - ref[key]) < 0.02, f"{key} khác với demo"

Cả ba mạng học được luật **bằng trắc** (90,5–94,0% câu đúng, so với 10,79% khi xáo trộn): đó là chuyện cục bộ, trong vài
âm tiết của một câu. Không câu nào chép nguyên văn Truyện Kiều.

Con số **xen kẽ 6↔8** cần đọc cẩn thận. Một mạng không nhớ gì về câu trước, chỉ chọn độ dài mỗi câu theo tỉ lệ riêng của
nó ($p_6$ câu 6 và $p_8$ câu 8 âm tiết), vẫn xen kẽ đúng với xác suất $2p_6p_8$ nhờ may rủi. Ở seed 0, LSTM vượt mức đó
khoảng 13 điểm phần trăm, GRU khoảng 5 điểm, còn RNN thì không vượt.

### 🤔 Dự đoán trước
Các con số trên chỉ là seed 0. Trong 3 seed của demo, có phải LSTM luôn nhớ nhịp 6–8 tốt hơn GRU không?

In [ ]:
LABEL = {
    ("lstm", 128): "LSTM",
    ("gru", 128): "GRU",
    ("rnn", 128): "RNN",
    ("gru", 150): "GRU 150",
    ("rnn", 280): "RNN 280",
}


def chance_alt(g):
    """2·p₆·p₈ từ biểu đồ tần suất số âm tiết mà demo lưu cho mỗi mẫu."""
    return 2 * g["hist"][6] * g["hist"][8] / g["lines"] ** 2


print(f"{'':>8} {'tham số':>8}   xen kẽ 6↔8 hơn mức may rủi (điểm %), seed 0 · 1 · 2")
for key, m in MODELS.items():
    gain = [100 * (s["gen"]["alt"] - chance_alt(s["gen"])) for s in m["seeds"]]
    print(f"{LABEL[key]:>8} {vn(m['params'], 0):>8}   " + " · ".join(f"{vn(v, 1):>5}" for v in gain))

**Không.** Seed 1 của GRU vượt mức may rủi 28,7 điểm, hơn mọi seed của LSTM; seed 2 của LSTM chỉ hơn 0,3 điểm, tức là gần
như không nhớ gì. Kết quả lấy mẫu dao động rất mạnh giữa các seed, vì 600 câu sinh ngẫu nhiên là một phép đo nhiễu, và vì
mỗi lần huấn luyện tìm ra (hoặc không) cách nhớ. Với $\eta$ chọn riêng cho từng loại mạng theo tập kiểm định, GRU
**không kém** LSTM ở thước đo này. Chung và cộng sự (2014) cũng thấy GRU và LSTM ngang nhau, khó nói bên nào hơn;
Jozefowicz và cộng sự (2015) thấy LSTM bắt kịp GRU khi cộng 1 vào $b_f$.

RNN thường thì ổn định ở một điểm: không seed nào vượt mức may rủi quá 4 điểm. Để thấy điều đó rõ hơn, ta cần một phép
đo ít nhiễu hơn việc lấy mẫu.

## 14 · Phép thử ít nhiễu: xuống dòng hay viết tiếp?

Đọc một câu **thật** của tập kiểm định (mạng được “mớm” đúng ký tự, không tự sinh), tới hết âm tiết thứ 6 mạng phải chọn:
**xuống dòng** (câu 6 âm tiết) hay **viết tiếp** bằng một dấu cách (câu 8). Sáu âm tiết đầu của hai loại câu theo cùng
một luật, nên chỉ có một cách đoán đúng: nhớ câu *trước* dài bao nhiêu. Tập kiểm định có đúng 127 câu mỗi loại, nên đoán
mò đúng 50%. Ô dưới đo xem thông tin cần nhớ nằm cách điểm quyết định bao xa. (Lúc học, mỗi đoạn 100 ký tự bắt đầu từ
$h = 0$, nên gradient đi ngược nhiều nhất 100 ký tự: đủ để học khoảng cách này.)

In [ ]:
def positions(line):
    """Với mỗi ký tự: thứ tự (1, 2, …) của âm tiết chứa nó trong câu; 0 cho ký tự xuống dòng."""
    pos, k, inword = [], 0, False
    for ch in line:
        if ch == "\n":
            pos.append(0)
            k, inword = 0, False
            continue
        word = ch.isalnum()
        if word and not inword:
            k += 1
        inword = word
        pos.append(k)
    return np.array(pos)


LAST6 = [max(i for i, k in enumerate(positions(ln)) if k == 6 and ln[i].isalnum()) for ln in VAL_LINES]
N_SYL = np.array([len(syllables(ln)) for ln in VAL_LINES])
end6 = [ln[-1] for ln in LINES if len(syllables(ln)) == 6]
gap = np.array(LAST6) + 2  # số ký tự từ dấu câu cuối câu trước tới điểm quyết định
print(f"tập kiểm định: {(N_SYL == 6).sum()} câu 6 và {(N_SYL == 8).sum()} câu 8 âm tiết → đoán mò đúng {pct(0.5, 0)}")
print(f"cả bài: {pct(np.mean([e == ',' for e in end6]), 1)} câu 6 âm tiết kết thúc bằng dấu phẩy")
print(f"dấu câu của câu trước nằm cách điểm quyết định {vn(gap.mean(), 1)} ký tự (từ {gap.min()} đến {gap.max()})")

Câu 6 âm tiết gần như luôn kết thúc bằng dấu phẩy, câu 8 bằng dấu chấm, chấm than hoặc chấm hỏi. Vậy điều cần nhớ chỉ
là **một bit** (câu trước là câu 6 hay câu 8), nhưng phải giữ nó qua khoảng 26 ký tự, gần bằng bài toán $T = 20$–40 của
mục 9. Ô dưới tính phép thử từ chính các logit đã thu ở mục 12, cho seed 0 của ba mạng, và so với demo.

In [ ]:
DECIDE = {}
for arch, (lg, _, _) in READ.items():
    start = np.cumsum([0] + [len(ln) + 1 for ln in VAL_LINES[:-1]])  # vị trí ký tự đầu mỗi câu trong tập kiểm định
    p_space = np.exp(log_softmax(lg[start + np.array(LAST6)])[:, IX[" "]])  # P(dấu cách): mạng chọn viết tiếp
    right = np.where(N_SYL == 8, p_space > 0.5, p_space < 0.5)
    DECIDE[arch] = right.mean()
    ref = MODELS[(arch, 128)]["seeds"][0]["decide"]
    print(
        f"{arch.upper():>4}: chọn đúng {pct(right.mean())} câu (demo {pct(ref['acc'])}); P(viết tiếp) trung bình: "
        f"câu 6 {vn(p_space[N_SYL == 6].mean(), 3)}, câu 8 {vn(p_space[N_SYL == 8].mean(), 3)}"
    )
    assert abs(right.mean() - ref["acc"]) < 0.005 and abs(p_space[N_SYL == 6].mean() - ref["space6"]) < 1e-3

Ở seed 0, LSTM chọn đúng 78,35% câu, GRU 65,35%, RNN 55,91%: gần mức đoán mò. Với câu 6, RNN vẫn cho xác suất viết
tiếp gần một nửa, tức là nó không biết câu trước dài bao nhiêu. Phép thử này dùng câu thật, nên không có nhiễu của việc
lấy mẫu; nhiễu còn lại đến từ chính lần huấn luyện. Hình dưới vẽ cả 3 seed của mọi mạng mà demo đã huấn luyện, kể cả GRU
và RNN to hơn, có số tham số xấp xỉ LSTM (150 và 280 đơn vị ẩn). Mỗi chấm là một seed; vạch xám là mức đoán mò 50%.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
STYLE = {"lstm": (BLUE, "o"), "gru": (GREEN, "^"), "rnn": (ORANGE, "s")}
keys = list(MODELS)
for i, key in enumerate(keys):
    color, marker = STYLE[key[0]]
    acc = [s["decide"]["acc"] for s in MODELS[key]["seeds"]]
    ax.scatter(100 * np.array(acc), [i] * 3, color=color, marker=marker, s=60, zorder=3, edgecolor="white")
    bits = " · ".join(vn(s["bits"], 3) for s in MODELS[key]["seeds"])
    print(
        f"{LABEL[key]:>8} ({vn(MODELS[key]['params'], 0)} tham số): chọn đúng " + " · ".join(pct(a, 1) for a in acc),
        f"  bit/ký tự {bits}",
    )
ax.axvline(50, color=GREY, ls="--", lw=1)
ax.set_yticks(range(len(keys)), [f"{LABEL[k]} ({vn(MODELS[k]['params'] / 1000, 0)} nghìn tham số)" for k in keys])
ax.invert_yaxis()
ax.set(xlim=(45, 95), xlabel="chọn đúng sau âm tiết 6 (%), 254 câu kiểm định")
ax.set_title("Nhớ được câu trước dài bao nhiêu? Mỗi chấm là một seed", fontsize=11)
ax.grid(axis="x", alpha=0.3)
vn_axes(fig)
plt.show()
rnn_all = [s["decide"]["acc"] for k, m in MODELS.items() if k[0] == "rnn" for s in m["seeds"]]
gated = [s["decide"]["acc"] for k, m in MODELS.items() if k[0] != "rnn" for s in m["seeds"]]
clear = max(rnn_all) + 0.05
print(f"6 seed RNN: {pct(min(rnn_all), 1)}–{pct(max(rnn_all), 1)}")
print(f"mạng có cổng vượt hẳn mọi seed RNN (trên {pct(clear, 1)}): {sum(a > clear for a in gated)}/{len(gated)} seed")

Đây là kết quả vững của bài:
- **Cả 6 seed RNN thường** nằm sát mức đoán mò (52,4–57,9%), kể cả RNN 280 đơn vị có số tham số bằng LSTM. Thêm đơn vị ẩn
  không cho RNN trí nhớ xa hơn; điều nó thiếu là một đường để gradient đi qua 26 ký tự.
- **Mạng có cổng** thì *có thể* nhớ: 5 trong 9 seed vượt hẳn mọi seed RNN, cao nhất 89,4%. Nhưng 4 seed còn lại thì ngang
  RNN. Cổng làm cho trí nhớ xa **học được**, không bảo đảm mỗi lần huấn luyện đều học được.
- Giữa LSTM và GRU, chênh lệch nằm **trong dao động giữa các seed**.

Bit/ký tự (cột cuối) xếp hạng ổn định hơn: cùng một seed, LSTM luôn thấp hơn GRU, và mọi seed RNN đều cao hơn mọi mạng
có cổng (RNN 280 đơn vị còn kém RNN 128). Thước đo này gồm chủ yếu những dự đoán cục bộ (chính tả, từ hay đi cùng nhau),
nên nó nói ít về trí nhớ xa.

## 15 · Nhìn vào một đơn vị: mạng đếm âm tiết

Muốn biết lúc nào xuống dòng, mạng phải biết mình đang ở âm tiết thứ mấy. Karpathy, Johnson & Fei-Fei (2015) tìm thấy
trong LSTM những ô nhớ theo dõi độ dài dòng, dấu ngoặc kép, dấu ngoặc. Ta làm phép tìm của demo: trong trạng thái đã thu
ở mục 12 (ô nhớ $c_t$ của LSTM, $h_t$ của hai mạng kia), tìm đơn vị có **tương quan** (hệ số Pearson $r$) lớn nhất với vị
trí âm tiết trong câu.

In [ ]:
pos_val = positions("".join(CHARS[i] for i in VAL))
keep = pos_val > 0  # bỏ các ký tự xuống dòng
TRACK = {}
for arch, (_, hs, cs) in READ.items():
    st = cs if arch == "lstm" else hs
    Z = st[keep] - st[keep].mean(0)
    y = pos_val[keep] - pos_val[keep].mean()
    r = Z.T @ y / np.sqrt((Z**2).sum(0) * (y**2).sum())
    unit = int(np.argmax(np.abs(r)))
    TRACK[arch] = (unit, st[:, unit], r[unit])
    ref = KIEU["ship"][arch]["tracker"]
    print(
        f"{arch.upper():>4}: đơn vị #{unit} của {'c' if arch == 'lstm' else 'h'}, r = {vn(r[unit], 3)} (demo: #{ref['unit']}, r = {vn(ref['r'], 3)})"
    )
    assert unit == ref["unit"] and abs(r[unit] - ref["r"]) < 1e-3, "khác với demo"

Cả ba trùng với demo. Hình trái vẽ đơn vị #51 trong ô nhớ của LSTM khi đọc bốn câu kiểm định đầu, từng ký tự (vạch gạch
xám: xuống dòng). Hình phải chuẩn hóa đơn vị tìm được của mỗi mạng (trừ trung bình, chia độ lệch chuẩn, đổi dấu nếu
$r < 0$) rồi lấy trung bình theo vị trí âm tiết trên cả tập kiểm định; mạng đếm tốt thì đường đi lên đều.

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.5), gridspec_kw={"width_ratios": [3, 2]})
unit, trace, _ = TRACK["lstm"]
n_show = sum(len(ln) + 1 for ln in VAL_LINES[:4])
a.plot(np.arange(n_show), trace[:n_show], color=BLUE, lw=1.5)
for i in np.flatnonzero(VAL[:n_show] == IX["\n"]):
    a.axvline(i, color=GREY, ls="--", lw=1)
a.set(xlabel="ký tự thứ … (4 câu kiểm định đầu)", ylabel=f"c[{unit}]", title=f"LSTM: ô nhớ #{unit} khi đọc từng ký tự")
for arch, (unit, trace, r) in TRACK.items():
    zs = np.sign(r) * (trace - trace[keep].mean()) / trace[keep].std()
    mean = [zs[pos_val == k].mean() for k in range(1, 9)]
    color, marker = STYLE[arch]
    b.plot(range(1, 9), mean, color=color, marker=marker, label=f"{arch.upper()} #{unit} (|r| = {vn(abs(r), 2)})")
b.set(xlabel="âm tiết thứ … trong câu", ylabel="giá trị chuẩn hóa", title="Trung bình trên 254 câu kiểm định")
b.legend(frameon=False, fontsize=9)
for ax in (a, b):
    ax.grid(alpha=0.3)
plt.tight_layout()
vn_axes(fig)
plt.show()
print(*VAL_LINES[:4], sep="\n")
line_len = np.concatenate([[len(syllables(ln))] * (len(ln) + 1) for ln in VAL_LINES])  # số âm tiết của câu chứa ký tự
for arch, (unit, trace, r) in TRACK.items():
    zs = np.sign(r) * (trace - trace[keep].mean()) / trace[keep].std()
    at = {n: [zs[(pos_val == k) & (line_len == n)].mean() for k in (1, 6)] for n in (6, 8)}
    print(
        f"{arch.upper():>4} #{unit}: âm tiết 1 → 6, câu 6: {vn(at[6][0], 2)} → {vn(at[6][1], 2)};"
        f" câu 8: {vn(at[8][0], 2)} → {vn(at[8][1], 2)}"
    )

Ô nhớ #51 của LSTM giảm đều theo từng âm tiết rồi bật lên lại ở mỗi lần xuống dòng: một **bộ đếm âm tiết**, không ai lập
trình cho nó, tự hình thành khi học đoán ký tự. Ở hình trái nó cũng xuống sâu hơn ở câu 8 âm tiết (câu 2 và 4) so với
câu 6. Dòng in cuối cho thấy vì sao: ngay từ âm tiết 1, giá trị chuẩn hóa của nó ở câu 8 đã khác ở câu 6, tức là nó mang
theo điều nhớ được từ câu trước. Đơn vị của GRU đếm gần tốt bằng. Đơn vị tốt nhất của RNN thì kém hẳn ($|r|$ = 0,66):
nó nhảy một lần ở âm tiết 4 rồi gần như đứng yên, và gần như không phân biệt câu 6 với câu 8. (Đây là đơn vị *tương quan
nhất*, không có nghĩa mạng chỉ dùng mỗi nó.)

## 16 · Thử thay đổi

Các ô dưới chạy lại cùng các hàm với tham số khác. Bạn có thể sửa giá trị rồi chạy lại.

**Khởi tạo cũng cứu được RNN thường?** Mục 5 cho thấy hệ số $g$ của $W_h$ đổi hẳn phần gradient về tới bước 0. Huấn luyện
RNN trên $T = 40$ (độ dài mà nó thất bại ở cả 3 seed của demo) với vài giá trị $g$, seed 0, 300 bước:

In [ ]:
for g in (1.0, 1.5, 2.0):
    acc = train_memory("rnn", 0.0, T=40, scale=g)[1]
    print(f"RNN, T = 40, g = {vn(g, 1)}: " + " ".join(pct(v, 0) for v in acc))

Với $g = 1{,}5$, gần ranh giới giữa co và bùng nổ (mục 5), RNN thường học được $T = 40$ chỉ sau vài trăm bước; với
$g = 2$ nó cũng học được, chậm hơn: gradient bùng nổ, và phép cắt ngưỡng giữ cho mỗi bước cập nhật không quá lớn. Khởi
tạo gần ranh giới đó là ý tưởng của khởi tạo trực giao hay đồng nhất cho RNN
([Le, Jaitly & Hinton 2015](https://arxiv.org/abs/1504.00941) đặt $W_h = I$ cho RNN dùng ReLU). Khởi tạo chỉ chọn điểm
xuất phát; cổng thì để mạng *tự học*, theo từng ký hiệu, lúc nào giữ và lúc nào quên. Thử thêm $T = 80$, hoặc các seed khác (`seed=1`).

**Nhiệt độ và câu mở đầu.** Đổi `temp` và `prompt` (chỉ dùng các ký tự có trong Truyện Kiều). Câu mở đầu dưới đây là câu
đầu tiên của bài thơ, nằm trong tập huấn luyện.

In [ ]:
prompt = "Trăm năm trong cõi người ta,\n"
for temp in (0.3, 0.8, 1.4):
    out = generate("lstm", LM["lstm"], prompt, 300, temp, np.random.default_rng(1)).split("\n")[:4]
    print(f"── LSTM, nhiệt độ {vn(temp, 1)}")
    for ln in out:
        print(f"  {ln:<44} {len(syllables(ln))} âm tiết  {tone_text(syllables(ln))}")

Nhiệt độ thấp cho câu tròn trịa nhưng dễ lặp các cụm quen; nhiệt độ cao làm vỡ chính tả và vỡ nhịp. Thử thêm GRU và RNN
(`"gru"`, `"rnn"`), hoặc tự gõ một câu mở đầu.

## 17 · Giới hạn và bước tiếp theo

Cổng giải quyết được một phần bài toán gradient, nhưng RNN còn hai giới hạn mà cổng không chữa được:

- **Trí nhớ có kích thước cố định.** Mọi điều mạng biết về quá khứ phải nén vào $h$ (và $c$): 128 số, dù đã đọc một câu hay
  cả bài thơ. Đo trên mô hình ngôn ngữ LSTM, Khandelwal và cộng sự (2018) thấy mô hình dùng khoảng 200 token ngữ cảnh,
  nhưng với những gì xa hơn khoảng 50 token thì nó gần như bỏ qua thứ tự từ, chỉ giữ một ý chung về chủ đề.
- **Tính tuần tự.** Bước $t$ cần $h_{t-1}$, nên không thể tính các bước song song: đọc 8.000 ký tự kiểm định là 8.000 phép
  tính nối đuôi nhau, dù máy có bao nhiêu nhân.

Với bài dịch máy, giới hạn thứ nhất thành nút thắt cổ chai: bộ mã hóa RNN phải ép cả câu nguồn vào một vectơ, và chất
lượng dịch tụt khi câu dài ra. Lời giải là để bộ giải mã **nhìn lại** mọi trạng thái của bộ mã hóa ở mỗi bước, thay vì
chỉ dựa vào vectơ cuối: cơ chế chú ý (attention). Đi xa hơn, Transformer bỏ hẳn vòng lặp, nên tính được mọi vị trí cùng
lúc. Ý tưởng cổng quên thì không mất đi: nó trở lại trong các mô hình tuyến tính như Mamba ở
[Bài 25](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/rnn-tro-lai/).

## Tóm tắt
- RNN dùng **cùng một bộ trọng số** ở mọi bước: $h_t = \tanh(x_tW_x + h_{t-1}W_h + b)$. BPTT là lan truyền ngược trên mạng
  trải theo thời gian, và gradient của $W_h$ là tổng qua mọi bước.
- Mỗi bước lùi, gradient nhân với $W_h^\top$ và $1 - h_t^2$: lúc khởi tạo, RNN 32 đơn vị đưa về được $3{,}7 \cdot 10^{-8}$
  gradient qua 80 bước; với $g = 2$ thì ngược lại, bùng nổ.
- Theo băng chuyền của LSTM, $\partial c_t/\partial c_{t-1} = f_t$. Nhưng với $b_f = 0$ thì $f_t \approx 0{,}5$, và LSTM còn
  giữ ít gradient hơn RNN; $b_f = 3$ (hoặc $b_z = 3$ cho GRU) giữ được 0,2 (0,06 với GRU) sau 80 bước.
- Bài “nhớ ký hiệu đầu” (3 seed, 5.000 bước): RNN và LSTM $b_f \le 1$ thất bại từ $T = 40$; $b_f = 3$, $b_z = 3$ nhớ được
  tới $T = 160$ ở 29/30 lần chạy. Khởi tạo quan trọng không kém kiến trúc.
- BPTT cắt ở $k = 10$ bước: RNN không học được phụ thuộc 20 bước.
- Truyện Kiều, mức ký tự: LSTM 2,2637, GRU 2,3412, RNN 2,4595 bit/ký tự (seed 0). Cả ba học được luật bằng trắc.
- Phép thử vững nhất là chọn xuống dòng sau âm tiết 6: mọi seed RNN 52,4–57,9% (mức đoán mò 50%), kể cả RNN 280 đơn vị;
  5/9 seed có cổng vượt hẳn, cao nhất 89,4%. LSTM và GRU chênh nhau trong dao động giữa các seed.

## Câu hỏi ôn tập
1. Vì sao số tham số của RNN không phụ thuộc vào độ dài chuỗi? Hệ quả của điều đó với gradient của $W_h$ là gì?
2. Gradient đi ngược qua một bước của RNN bị nhân với những gì? Vì sao cắt ngưỡng gradient chữa được bùng nổ mà không
   chữa được biến mất?
3. LSTM có băng chuyền ô nhớ, vậy mà với $b_f = 0$ nó giữ gradient kém hơn RNN thường. Giải thích, và nêu cách sửa.
4. Với BPTT cắt ở $k = 10$, RNN không học được bài $T = 20$, nhưng LSTM $b_f = 3$ lên được khoảng 78%. Vì sao?
5. Seed 1 của GRU xen kẽ 6↔8 tốt hơn mọi seed LSTM. Có thể kết luận GRU tốt hơn LSTM không? Phép đo nào đáng tin hơn,
   và vì sao?
6. RNN 280 đơn vị ẩn có số tham số bằng LSTM nhưng vẫn chỉ chọn đúng sau âm tiết 6 ở mức gần đoán mò. Điều đó nói gì về
   trí nhớ của RNN?

<details><summary>Gợi ý đáp án</summary>

1. Mọi bước dùng chung $W_x$, $W_h$, $b$. Vì vậy gradient của $W_h$ là tổng các đóng góp $h_{t-1}^\top\delta_t$ của mọi
   bước, và đóng góp của các bước xa bị nhân với nhiều thừa số trên đường về.
2. Với $W_h^\top$ và với đạo hàm tanh $1 - h_t^2 \le 1$. Cắt ngưỡng thu nhỏ một gradient quá lớn về độ lớn cho phép mà giữ
   hướng; còn một gradient đã nhỏ tới $10^{-8}$ thì phần đến từ bước xa đã chìm dưới phần từ các bước gần, phóng to cả
   vectơ không làm nó nổi lên.
3. Với $b_f = 0$, $f_t \approx \sigma(0) = 0{,}5$, nên mỗi bước băng chuyền giữ một nửa: $0{,}5^{80} \approx 10^{-24}$; các
   đường còn lại đi qua $W_h$ và thêm các thừa số sigmoid. Sửa: cộng 1 hoặc hơn vào $b_f$ lúc khởi tạo.
4. Với $f_t \approx 0{,}95$ ngay từ đầu, ô nhớ chưa học vẫn mang một phần ký hiệu đầu tới cuối chuỗi; lớp đọc ra chỉ cần
   gradient ở bước cuối để học khai thác nó. Cắt ngắn chặn việc học cách nhớ, không chặn trí nhớ có sẵn.
5. Không. Ba seed của mỗi mạng dao động rất mạnh (GRU: 0,1 đến 28,7 điểm), và chênh lệch giữa LSTM và GRU nằm trong dao
   động đó. Phép thử xuống dòng trên câu thật đáng tin hơn vì không có nhiễu của việc lấy mẫu, và nó tách rõ RNN (mọi
   seed sát 50%) khỏi các mạng có cổng.
6. Thêm đơn vị ẩn không giúp gradient đi qua khoảng 26 ký tự giữa dấu câu của câu trước và điểm quyết định; thiếu một
   đường tắt như băng chuyền thì mạng không học được trí nhớ xa, dù trạng thái to bao nhiêu.
</details>

## Tìm hiểu thêm
- J. Elman (1990), [Finding Structure in Time](https://doi.org/10.1207/s15516709cog1402_1): mạng hồi tiếp đơn giản.
- S. Hochreiter & J. Schmidhuber (1997), [Long Short-Term Memory](https://doi.org/10.1162/neco.1997.9.8.1735);
  F. Gers, J. Schmidhuber & F. Cummins (2000), [Learning to Forget](https://doi.org/10.1162/089976600300015015): cổng quên.
- R. Pascanu, T. Mikolov & Y. Bengio (2013), [On the difficulty of training recurrent neural networks](https://arxiv.org/abs/1211.5063):
  điều kiện gradient biến mất và bùng nổ, cắt ngưỡng gradient.
- K. Cho và cộng sự (2014), [Learning Phrase Representations using RNN Encoder–Decoder](https://arxiv.org/abs/1406.1078)
  (§2.3: đơn vị có cổng); J. Chung và cộng sự (2014), [Empirical Evaluation of Gated Recurrent Neural Networks](https://arxiv.org/abs/1412.3555).
- R. Jozefowicz, W. Zaremba & I. Sutskever (2015), [An Empirical Exploration of Recurrent Network Architectures](https://proceedings.mlr.press/v37/jozefowicz15.html):
  cộng 1 vào độ lệch cổng quên.
- A. Karpathy, J. Johnson & L. Fei-Fei (2015), [Visualizing and Understanding Recurrent Networks](https://arxiv.org/abs/1506.02078):
  các ô nhớ theo dõi độ dài dòng, dấu ngoặc kép.
- U. Khandelwal và cộng sự (2018), [Sharp Nearby, Fuzzy Far Away](https://arxiv.org/abs/1805.04623): LSTM dùng bao
  nhiêu ngữ cảnh.
- C. Olah (2015), [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/);
  A. Karpathy (2015), [The Unreasonable Effectiveness of Recurrent Neural Networks](https://karpathy.github.io/2015/05/21/rnn-effectiveness/).
- d2l.ai, [Backpropagation Through Time](https://d2l.ai/chapter_recurrent-neural-networks/bptt.html): BPTT đầy đủ, cắt
  ngắn đều và cắt ngắn ngẫu nhiên.
- Nguyễn Du, [Truyện Kiều](https://vi.wikisource.org/wiki/Truy%E1%BB%87n_Ki%E1%BB%81u) trên Wikisource (bản sửa 205132).

**Bài tiếp theo:** [Bài 16 · Cơ chế chú ý (Attention)](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/attention/): thay
vì nén cả câu vào một trạng thái, nhìn lại đúng chỗ cần.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")